# DuckPD: Temporal Semantics, Fixed-Duration Rolling Windows & Categoricals

This notebook focuses on DuckPD's temporal, ordered-window, and categorical contracts:
1. **Out-of-Core Data Ingestion & Setup**: Preserving file scan order and configuring session-level execution.
2. **Fixed-Duration Rolling Windows (`DataFrame.rolling` & `GroupBy.rolling`)**: Nanosecond `RANGE` frames with string/timedelta durations, boundary options (`closed='right'|'left'|'both'|'neither'`), and pandas-compatible defaults.
3. **Lazy Temporal Accessors & Arithmetic (`Series.dt`)**: Fixed-duration `floor()`, `ceil()`, and `round()`, timezone conversion (`tz_convert`), UTC localization (`tz_localize`), and timestamp/duration arithmetic (`+`, `-`, comparisons).
4. **Categorical Semantics (`Series.cat`) & Unused-Category Groupbys**: Categorical metadata preservation, `cat.codes`, `as_ordered()`, ordered comparisons, and `groupby(observed=False)` aggregation expansions.
5. **Direct SQL Execution & Lazy Expression Chaining**: Seamless blending between DuckPD DataFrames, DuckDB SQL queries, and logical plan inspection.
6. **Zero-Copy Arrow & Pandas Interoperability**: Collecting results into Pandas and PyArrow structures while verifying semantic correctness and execution counts.

## 1. Environment Setup and DuckPD Initialization

We initialize a DuckPD session, inspect session configurations, and verify zero eager executions during registration.

In [ ]:
from datetime import timedelta
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

import duckpd

# Connect to a DuckPD session
session = duckpd.connect()
print(f"DuckPD version: {duckpd.__version__}")
print(f"Initial session execution count: {session.execution_count}")

## 2. Out-of-Core Data Ingestion (Parquet and CSV)

DuckPD scans Parquet and CSV files lazily without loading entire datasets into memory. Starting in recent versions, CSV and Parquet readers preserve file scan order automatically via hidden stable row identities.

Let's generate a time-series dataset of financial orders across multiple asset classes and write it to Parquet and CSV.

In [ ]:
temp_dir = TemporaryDirectory()
work_dir = Path(temp_dir.name)
parquet_path = work_dir / "market_orders.parquet"
csv_path = work_dir / "market_orders.csv"

# Construct a realistic multi-asset trade & order dataset with timestamps and categories
base_time = pd.Timestamp("2024-03-10 09:30:00", tz="UTC")
timestamps = pd.Series(
    [base_time + pd.Timedelta(minutes=5 * i) for i in range(12)], dtype="datetime64[ns, UTC]"
)

raw_df = pd.DataFrame(
    {
        "order_id": [f"ORD_{1000 + i}" for i in range(12)],
        "timestamp": timestamps,
        "symbol": [
            "AAPL",
            "NVDA",
            "AAPL",
            "GOOG",
            "NVDA",
            "AAPL",
            "GOOG",
            "AAPL",
            "NVDA",
            "GOOG",
            "AAPL",
            "NVDA",
        ],
        "asset_class": pd.Categorical(
            [
                "Equity",
                "Equity",
                "Equity",
                "Equity",
                "Equity",
                "Equity",
                "Equity",
                "Equity",
                "Equity",
                "Equity",
                "Equity",
                "Equity",
            ],
            categories=["Equity", "Fixed_Income", "Commodity", "Crypto"],
            ordered=True,
        ),
        "execution_venue": pd.Categorical(
            [
                "NASDAQ",
                "NASDAQ",
                "NASDAQ",
                "BATS",
                "NASDAQ",
                "ARCA",
                "BATS",
                "NASDAQ",
                "ARCA",
                "BATS",
                "ARCA",
                "NASDAQ",
            ],
            categories=["NASDAQ", "ARCA", "BATS", "IEX", "DARK_POOL"],
            ordered=False,
        ),
        "price": [
            182.50,
            895.00,
            183.10,
            142.20,
            899.50,
            182.90,
            143.00,
            184.00,
            902.10,
            142.75,
            183.80,
            905.00,
        ],
        "quantity": [150, 40, 200, 300, 50, 180, 250, 350, 60, 400, 220, 75],
        "latency_ms": [12, 18, 9, 25, 15, 11, 30, 8, 14, 28, 10, 16],
    }
)

# Write to disk
raw_df.to_parquet(parquet_path, index=False)
raw_df.to_csv(csv_path, index=False)

# Lazily scan parquet and csv out-of-core
orders_pq = session.read_parquet(parquet_path, order_by="timestamp")
orders_csv = session.read_csv(csv_path)

# Retain pandas source snapshot with full categorical metadata and ordering
orders = session.from_pandas(raw_df, order_by="timestamp")

print(f"Scanned Parquet frame columns: {orders_pq.columns}")
print(f"Scanned CSV frame columns: {orders_csv.columns}")
print(f"Orders frame columns: {orders.columns}")
print(f"Eager executions so far (should remain 0): {session.execution_count}")

## 3. Lazy Filtering and Expression Chaining

DuckPD builds an optimized logical plan without executing until required (`head()`, `collect()`, `write_parquet()`, etc.).
Here we showcase:
- **Timestamp and Duration Arithmetic**: Adding/subtracting `timedelta` objects directly on Series.
- **Fixed-Duration Temporal Rounding**: `Series.dt.floor()`, `ceil()`, and `round()` using positive duration frequencies (e.g., `'15min'`, `'1h'`).
- **Timezone Operations**: `tz_convert('America/New_York')` for timezone awareness, and `tz_localize(None)` for stripping timezone metadata.
- **Categorical Accessors & Ordered Filtering**: Checking `.cat.categories`, `.cat.codes`, and comparing ordered categoricals.

In [ ]:
# 1. Inspect categorical metadata directly on lazy Series
print("Venue categories:", orders["execution_venue"].cat.categories.tolist())
print("Venue is ordered:", orders["execution_venue"].cat.ordered)
print("Asset class is ordered:", orders["asset_class"].cat.ordered)

# 2. Chain expressions: temporal arithmetic, rounding, tz conversion, and categorical codes
transformed = orders[orders["price"] > 150.0].assign(
    # Duration arithmetic: project trade settlement timestamp (+2 hours)
    settlement_time=lambda df: df["timestamp"] + timedelta(hours=2),
    # Fixed duration rounding: bucket into 15-minute intervals
    time_15m_floor=lambda df: df["timestamp"].dt.floor("15min"),
    time_15m_ceil=lambda df: df["timestamp"].dt.ceil("15min"),
    time_1h_round=lambda df: df["timestamp"].dt.round("1h"),
    # Timezone conversion: convert UTC to Eastern Wall Clock
    ny_time=lambda df: df["timestamp"].dt.tz_convert("America/New_York"),
    # Extract integer category codes lazily
    venue_code=lambda df: df["execution_venue"].cat.codes,
    # Dollar trade notional value
    notional=lambda df: df["price"] * df["quantity"],
)

print(f"Executions before preview/collect: {session.execution_count}")

# Preview top 5 rows using head()
preview = transformed[
    ["order_id", "symbol", "timestamp", "ny_time", "time_15m_floor", "notional", "venue_code"]
].head(5)
preview

In [ ]:
# Show the compiled SQL without the verbose physical-plan dump.
print(transformed.explain(mode="sql"))

## 4. Advanced Window Functions and Grouped Aggregations

This workflow combines:
1. **Fixed-Duration Rolling Windows**: `DataFrame.rolling("30min", on="timestamp")` and `DataFrameGroupBy.rolling(...)` compiling to nanosecond `RANGE` frames. Supports boundary semantics (`closed='right'|'left'|'both'|'neither'`) and `min_periods`.
2. **Categorical `groupby(observed=False)` Aggregations**: Automatically expanding unused categories in categorical group keys (e.g. keeping unrepresented venues or asset classes in the summary output).

In [ ]:
# 1. Global fixed-duration rolling 20-minute window with closed='right'
rolling_global = orders.rolling("20min", on="timestamp", closed="right").mean(numeric_only=True)
print("Global 20-minute rolling means (numeric columns):")
print(rolling_global.collect().head(6))

# 2. Per-symbol fixed-duration rolling 30-minute window
# Group keys compile to window partitions, and timestamps define the range frame
rolling_by_symbol = (
    orders.groupby("symbol")
    .rolling("30min", on="timestamp", closed="right")
    .mean(numeric_only=True)
)
print("\nPer-symbol 30-minute rolling means:")
rolling_by_symbol.collect()

In [ ]:
# Categorical Groupby with observed=False
# This preserves and expands all declared categories in the output, even if no rows exist for them!
venue_summary = orders.groupby("execution_venue", observed=False, as_index=True).agg(
    order_count=("quantity", "count"),
    total_shares=("quantity", "sum"),
    avg_price=("price", "mean"),
)

print("Venue aggregation with observed=False (IEX and DARK_POOL are preserved with nulls/zeros):")
venue_df = venue_summary.collect()
venue_df

## 5. Direct SQL Execution on DuckPD DataFrames

You can run raw DuckDB SQL queries directly against existing DuckPD DataFrame objects using `session.sql()`. DuckPD registers or references the underlying relations seamlessly, allowing you to interleave SQL analytics and DataFrame API transformations.

In [ ]:
# Run DuckDB SQL directly on the Parquet dataset using session.sql()
sql_query = f"""
    SELECT
        symbol,
        count(*) AS trade_count,
        round(avg(price), 2) AS vwap,
        min(timestamp) AS first_trade,
        max(timestamp) AS last_trade
    FROM read_parquet('{parquet_path}')
    GROUP BY symbol
    ORDER BY trade_count DESC, symbol ASC
"""

sql_frame = session.sql(sql_query)
print("Result of direct DuckDB SQL query:")
sql_result = sql_frame.collect()
sql_result

## 6. Zero-Copy Conversion to Apache Arrow and Pandas

DuckPD integrates smoothly with downstream ecosystems:
- Collect results as standard `pandas.DataFrame` or `pandas.Series` with exact categorical dtypes and timezone-aware timestamps preserved.
- Collect zero-copy `pyarrow.Table` objects via `to_arrow()`.
- Export directly to Parquet (`write_parquet()`) or CSV (`write_csv()`) directly inside DuckDB without materializing full intermediate tables in Python memory.

In [ ]:
# 1. Zero-copy export to PyArrow Table
arrow_table = transformed.to_arrow()
print(f"Exported PyArrow Table type: {type(arrow_table)}")
print(f"Schema:\n{arrow_table.schema}")
print(f"PyArrow Row Count: {arrow_table.num_rows}")

# 2. Collect as Pandas DataFrame and verify categorical and temporal metadata
collected_pandas = transformed.collect()
print("\nPandas DataFrame dtypes:")
print(collected_pandas.dtypes)

# Verify categorical metadata round-trip
assert isinstance(collected_pandas["execution_venue"].dtype, pd.CategoricalDtype)
assert list(collected_pandas["execution_venue"].cat.categories) == [
    "NASDAQ",
    "ARCA",
    "BATS",
    "IEX",
    "DARK_POOL",
]

# 3. Direct out-of-core write to Parquet
output_parquet = work_dir / "processed_market_orders.parquet"
transformed.write_parquet(output_parquet)
print(f"\nWritten out-of-core Parquet file size: {output_parquet.stat().st_size} bytes")

# Clean up temporary directory
temp_dir.cleanup()
print("\nCompleted successfully! Final session execution count:", session.execution_count)